<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h1 style="font-size: 26px; margin-bottom: 8px;">Object-Oriented Programming in Python: Complete Practical Guide</h1>
<p style="font-size: 15px;">This notebook covers modular OOP patterns, validation, typing, and helper libraries with practical edge cases.</p>
</div>

In [1]:
import asyncio
import json
import logging
import uuid
from abc import ABC, abstractmethod
from collections import Counter, defaultdict
from dataclasses import dataclass
from functools import lru_cache
from typing import Any, Dict, List, Literal, Optional, Protocol, TypedDict, Union

from pydantic import BaseModel, Field, ValidationError

print("All imports loaded")

All imports loaded


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">1) Core OOP in Python</h2>
<p style="font-size: 15px;">Encapsulation, inheritance, polymorphism, class methods, static methods, and edge-case handling.</p>
</div>

In [2]:
class BankAccount:
    bank_name = "PyBank"

    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self._balance = 0.0
        self.balance = balance

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if value < 0:
            raise ValueError("Balance cannot be negative")
        self._balance = float(value)

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdraw must be positive")
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount

    @classmethod
    def create_with_bonus(cls, owner: str, bonus: float) -> "BankAccount":
        return cls(owner, max(0.0, bonus))

    @staticmethod
    def is_valid_owner(name: str) -> bool:
        return bool(name and name.strip())


class PremiumAccount(BankAccount):
    def withdraw(self, amount: float) -> None:
        if amount > self.balance + 50:
            raise ValueError("Overdraft limit exceeded")
        self._balance -= amount


base = BankAccount("Asha", 100)
base.deposit(20)
base.withdraw(30)
print(base.balance)

premium = PremiumAccount.create_with_bonus("Ravi", 10)
premium.withdraw(55)
print(premium.balance)

for case in [lambda: base.deposit(0), lambda: base.withdraw(9999)]:
    try:
        case()
    except ValueError as exc:
        print("Edge case:", exc)

90.0
-45.0
Edge case: Deposit must be positive
Edge case: Insufficient funds


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">2) pydantic — Data Validation with BaseModel</h2>
<p style="font-size: 15px;">
Pydantic enforces runtime data validation using Python type hints. Use <code>BaseModel</code> to declare schemas, <code>Field</code> for constraints, and <code>ValidationError</code> to inspect failures. Supports nested models and Pydantic v1/v2.
</p>
</div>

In [3]:
# ── pydantic: BaseModel, Field, ValidationError ──────────────────────────────

class Answer(BaseModel):
    question: str
    score: float = Field(..., ge=0.0, le=1.0, description="Score between 0 and 1")

class QASession(BaseModel):
    session_id: str
    answers: List[Answer]
    metadata: Optional[Dict[str, Any]] = None

def to_dict(model: BaseModel) -> Dict[str, Any]:
    """Supports both Pydantic v1 (.dict) and v2 (.model_dump)."""
    return model.model_dump() if hasattr(model, "model_dump") else model.dict()

# ── Valid models ──────────────────────────────────────────────────────────────
valid = Answer(question="What is OOP?", score=0.9)
print("Single model:", to_dict(valid))

session = QASession(
    session_id="s1",
    answers=[Answer(question="Q1", score=0.8), Answer(question="Q2", score=0.5)],
    metadata={"source": "exam"},
)
print("Nested model:", to_dict(session))

# ── Edge cases: validation failures ──────────────────────────────────────────
for bad in [{"question": "over", "score": 1.5}, {"question": "under", "score": -0.1}, {"question": "missing"}]:
    try:
        Answer(**bad)
    except ValidationError as exc:
        print("ValidationError:", exc.errors()[0]["msg"], "| input:", bad)

Single model: {'question': 'What is OOP?', 'score': 0.9}
Nested model: {'session_id': 's1', 'answers': [{'question': 'Q1', 'score': 0.8}, {'question': 'Q2', 'score': 0.5}], 'metadata': {'source': 'exam'}}
ValidationError: Input should be less than or equal to 1 | input: {'question': 'over', 'score': 1.5}
ValidationError: Input should be greater than or equal to 0 | input: {'question': 'under', 'score': -0.1}
ValidationError: Field required | input: {'question': 'missing'}


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">3) typing — Type Hints and Structured Annotations</h2>
<p style="font-size: 15px;">
The <code>typing</code> module enables static type checking and expressive function signatures. Key exports: <code>List</code>, <code>Dict</code>, <code>Optional</code>, <code>Union</code>, <code>Literal</code>, <code>Any</code>, and <code>TypedDict</code>.
</p>
</div>

In [4]:
# ── typing: Type Hints and Structured Annotations ────────────────────────────

class UserPayload(TypedDict, total=False):
    """total=False means all keys are optional."""
    id: int
    name: str
    role: Literal["admin", "viewer", "editor"]

def normalize_tags(tags: Optional[List[str]]) -> List[str]:
    """Strip, lowercase, and drop blank tags. Returns [] for None input."""
    return [t.strip().lower() for t in tags if t.strip()] if tags else []

def resolve_value(value: Union[int, float, str]) -> str:
    return f"resolved:{value!r}"

def describe_payload(payload: UserPayload) -> Dict[str, Any]:
    return {
        "has_id": "id" in payload,
        "name": payload.get("name", "unknown"),
        "role": payload.get("role", "viewer"),
    }

def process_any(value: Any) -> str:
    return f"type={type(value).__name__!r}, repr={value!r}"

# ── Demos ─────────────────────────────────────────────────────────────────────
print(normalize_tags(["  ML ", "Python", "", "OOP"]))  # strips blanks
print(normalize_tags(None))                            # None → []
print(normalize_tags([]))                              # empty → []

print(resolve_value(3.14))
print(resolve_value("hello"))

print(describe_payload({"name": "Nina", "role": "admin"}))
print(describe_payload({}))                            # all optional, edge case

print(process_any(None))
print(process_any([1, "two", 3.0]))

['ml', 'python', 'oop']
[]
[]
resolved:3.14
resolved:'hello'
{'has_id': False, 'name': 'Nina', 'role': 'admin'}
{'has_id': False, 'name': 'unknown', 'role': 'viewer'}
type='NoneType', repr=None
type='list', repr=[1, 'two', 3.0]


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">4) json — Parsing and Serializing Data</h2>
<p style="font-size: 15px;">
Use <code>json.loads</code> to parse strings and <code>json.dumps</code> to serialize Python objects. Always handle <code>JSONDecodeError</code> and validate the expected root type (dict vs list).
</p>
</div>

In [5]:
# ── json: Parsing, Serializing, and Error Handling ───────────────────────────

def parse_response(response: str) -> Dict[str, Any]:
    """Safely parse a JSON string; return structured error if it fails."""
    try:
        data = json.loads(response)
    except json.JSONDecodeError as exc:
        return {"ok": False, "error": f"JSONDecodeError at pos {exc.pos}: {exc.msg}"}
    if not isinstance(data, dict):
        return {"ok": False, "error": f"Expected JSON object, got {type(data).__name__}"}
    return {"ok": True, "data": data}

def safe_serialize(obj: Any, indent: int = 2) -> str:
    """Serialize arbitrary objects; fall back to str() for non-serializable types."""
    try:
        return json.dumps(obj, indent=indent, default=str)
    except (TypeError, ValueError) as exc:
        return f"Serialize error: {exc}"

# ── Valid JSON ────────────────────────────────────────────────────────────────
print(parse_response('{"status": "ok", "value": 42}'))

# ── Edge cases ────────────────────────────────────────────────────────────────
print(parse_response('{status: ok}'))        # malformed
print(parse_response('[1, 2, 3]'))           # list instead of dict
print(parse_response(''))                    # empty string

# ── Serialization (default=str handles non-serializable types) ────────────────
import datetime
obj = {"name": "Alice", "score": 0.9, "created": datetime.datetime.now()}
print(safe_serialize(obj))

{'ok': True, 'data': {'status': 'ok', 'value': 42}}
{'ok': False, 'error': 'JSONDecodeError at pos 1: Expecting property name enclosed in double quotes'}
{'ok': False, 'error': 'Expected JSON object, got list'}
{'ok': False, 'error': 'JSONDecodeError at pos 0: Expecting value'}
{
  "name": "Alice",
  "score": 0.9,
  "created": "2026-04-23 06:52:59.816579"
}


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">5) collections — defaultdict and Counter</h2>
<p style="font-size: 15px;">
<code>defaultdict</code> prevents KeyError by auto-initialising missing keys. <code>Counter</code> provides efficient frequency counting with set-like arithmetic.
</p>
</div>

In [6]:
# ── collections: defaultdict ──────────────────────────────────────────────────

def group_scores(rows: List[Dict[str, Union[str, int]]]) -> Dict[str, List[int]]:
    grouped: Dict[str, List[int]] = defaultdict(list)
    for row in rows:
        grouped[str(row["topic"])].append(int(row["score"]))
    return dict(grouped)

records = [
    {"topic": "math", "score": 80},
    {"topic": "python", "score": 95},
    {"topic": "math", "score": 88},
    {"topic": "python", "score": 72},
]
grouped = group_scores(records)
print("Grouped:", grouped)
print("Averages:", {k: round(sum(v) / len(v), 1) for k, v in grouped.items()})

# ── collections: Counter ──────────────────────────────────────────────────────

def word_frequency(text: str) -> Counter:
    tokens = [t.strip(".,!?;:").lower() for t in text.split() if t.strip()]
    return Counter(tokens)

freq = word_frequency("Python python OOP makes Python fun and OOP practical")
print("\nFrequency:", freq)
print("Top 2:", freq.most_common(2))

# Counter arithmetic (set-style subtraction removes zero/negative counts)
a = Counter({"python": 5, "oop": 3, "ml": 1})
b = Counter({"python": 2, "oop": 3})
print("a - b:", a - b)          # only positives survive
print("a & b:", a & b)          # intersection (min counts)
print("a | b:", a | b)          # union (max counts)

Grouped: {'math': [80, 88], 'python': [95, 72]}
Averages: {'math': 84.0, 'python': 83.5}

Frequency: Counter({'python': 3, 'oop': 2, 'makes': 1, 'fun': 1, 'and': 1, 'practical': 1})
Top 2: [('python', 3), ('oop', 2)]
a - b: Counter({'python': 3, 'ml': 1})
a & b: Counter({'oop': 3, 'python': 2})
a | b: Counter({'python': 5, 'oop': 3, 'ml': 1})


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">6) uuid — Unique Identifiers</h2>
<p style="font-size: 15px;">
<code>uuid4()</code> generates a random UUID; <code>uuid5()</code> produces a deterministic one from a namespace + name. Always validate user-supplied UUIDs before use.
</p>
</div>

In [8]:
# ── uuid: Unique Identifiers ──────────────────────────────────────────────────

def safe_parse_uuid(raw: str) -> Optional[uuid.UUID]:
    """Return a UUID object or None for any invalid / non-string input."""
    try:
        return uuid.UUID(raw)
    except (ValueError, AttributeError, TypeError):
        return None

# ── UUID versions ─────────────────────────────────────────────────────────────
u1 = uuid.uuid1()                               # host MAC + timestamp
u4 = uuid.uuid4()                               # random (most common)
u5 = uuid.uuid5(uuid.NAMESPACE_DNS, "example.org")  # deterministic/name-based

print("uuid1 (host+time):", u1)
print("uuid4 (random):   ", u4)
print("uuid5 (name-based):", u5)

# uuid5 is deterministic — same name always produces the same UUID
print("uuid5 deterministic:", u5 == uuid.uuid5(uuid.NAMESPACE_DNS, "example.org"))

# ── Parsing ───────────────────────────────────────────────────────────────────
print("Parsed valid:", safe_parse_uuid(str(u4)))
print("Parsed invalid:", safe_parse_uuid("not-a-uuid"))
print("Parsed None:   ", safe_parse_uuid(None))  # TypeError handled → None

# ── Useful attributes ─────────────────────────────────────────────────────────
print("hex:", u4.hex, "| version:", u4.version, "| int:", str(u4.int)[:20], "...")

uuid1 (host+time): 0e3bec6a-3eb3-11f1-bdc5-78af08c5c5a5
uuid4 (random):    108bbcd1-7a4d-43de-b1aa-4e8c114a35a5
uuid5 (name-based): aad03681-8b63-5304-89e0-8ca8f49461b5
uuid5 deterministic: True
Parsed valid: 108bbcd1-7a4d-43de-b1aa-4e8c114a35a5
Parsed invalid: None
Parsed None:    None
hex: 108bbcd17a4d43deb1aa4e8c114a35a5 | version: 4 | int: 21993206885437025994 ...


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">7) asyncio — Concurrent Workflows</h2>
<p style="font-size: 15px;">
<code>asyncio</code> enables cooperative concurrency using <code>async/await</code>. Use <code>asyncio.gather</code> to run tasks in parallel, <code>return_exceptions=True</code> to handle partial failures, and <code>asyncio.wait_for</code> for timeouts. Works natively in Jupyter with <code>await</code> at cell level.
</p>
</div>

In [14]:
# ── asyncio: async/await, gather, timeout, return_exceptions ─────────────────

async def fetch(name: str, delay: float, fail: bool = False) -> str:
    """Simulates an async network call."""
    await asyncio.sleep(delay)
    if fail:
        raise ConnectionError(f"{name}: connection refused")
    return f"{name}: fetched"

async def pipeline() -> List[Union[str, Exception]]:
    """Run multiple async tasks; failures do not cancel others."""
    return await asyncio.gather(
        fetch("ServiceA", 0.05),
        fetch("ServiceB", 0.08, fail=True),
        fetch("ServiceC", 0.02),
        return_exceptions=True,
    )

# Jupyter supports top-level await
results = await asyncio.wait_for(pipeline(), timeout=1.0)
for r in results:
    tag = "OK  " if isinstance(r, str) else "FAIL"
    print(f"[{tag}] {r}")

# ── Edge case: timeout exceeded ───────────────────────────────────────────────
async def slow_call() -> str:
    await asyncio.sleep(5)
    return "too late"

try:
    await asyncio.wait_for(slow_call(), timeout=0.05)
except asyncio.TimeoutError:
    print("[TIMEOUT] slow_call exceeded deadline")

[OK  ] ServiceA: fetched
[FAIL] ServiceB: connection refused
[OK  ] ServiceC: fetched
[TIMEOUT] slow_call exceeded deadline


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">8) logging — Structured Runtime Logs</h2>
<p style="font-size: 15px;">
Use the standard <code>logging</code> module rather than <code>print</code> for production code. Supports levels (DEBUG → INFO → WARNING → ERROR → CRITICAL), formatters, and multiple handlers.
</p>
</div>

In [10]:
# ── logging: Structured Runtime Logs ─────────────────────────────────────────

def build_logger(name: str, level: int = logging.DEBUG) -> logging.Logger:
    """Create a named logger with a consistent format."""
    log = logging.getLogger(name)
    if not log.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(
            logging.Formatter("%(asctime)s | %(name)s | %(levelname)-8s | %(message)s")
        )
        log.addHandler(handler)
    log.setLevel(level)
    return log


class DataProcessor:
    def __init__(self, name: str) -> None:
        self.log = build_logger(name)

    def process(self, records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        self.log.info("Starting — %d records received", len(records))
        results = []
        for rec in records:
            if "value" not in rec:
                self.log.warning("Skipping record missing 'value': %s", rec)
                continue
            if not isinstance(rec["value"], (int, float)):
                self.log.error("Invalid type for 'value' in record: %s", rec)
                continue
            results.append({**rec, "processed": True})
        self.log.info("Done — %d/%d records processed", len(results), len(records))
        return results


proc = DataProcessor("DataProcessor")
out = proc.process([
    {"id": 1, "value": 42},
    {"id": 2},                      # missing 'value'
    {"id": 3, "value": "bad_type"}, # wrong type
    {"id": 4, "value": 99},
])
print("Result:", out)

2026-04-23 06:53:47,102 | DataProcessor | INFO     | Starting — 4 records received
2026-04-23 06:53:47,106 | DataProcessor | WARNING  | Skipping record missing 'value': {'id': 2}
2026-04-23 06:53:47,110 | DataProcessor | ERROR    | Invalid type for 'value' in record: {'id': 3, 'value': 'bad_type'}
2026-04-23 06:53:47,114 | DataProcessor | INFO     | Done — 2/4 records processed


Result: [{'id': 1, 'value': 42, 'processed': True}, {'id': 4, 'value': 99, 'processed': True}]


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">9) abc — Abstract Base Classes</h2>
<p style="font-size: 15px;">
<code>ABC</code> + <code>@abstractmethod</code> define contracts that subclasses must fulfil. Attempting to instantiate an abstract class raises <code>TypeError</code>. Use them for plugin architectures, parser interfaces, and service boundaries.
</p>
</div>

In [11]:
# ── abc: Abstract Base Classes ────────────────────────────────────────────────

class Parser(ABC):
    """Contract: every concrete parser MUST implement parse()."""

    @abstractmethod
    def parse(self, text: str) -> Dict[str, Any]:
        pass

    def safe_parse(self, text: str) -> Dict[str, Any]:
        """Template method: wraps parse() with error handling."""
        try:
            return self.parse(text)
        except Exception as exc:
            return {"error": str(exc), "raw": text[:80]}


class JsonParser(Parser):
    def parse(self, text: str) -> Dict[str, Any]:
        payload = json.loads(text)
        if not isinstance(payload, dict):
            raise ValueError(f"Expected dict, got {type(payload).__name__}")
        return payload


class CsvParser(Parser):
    def parse(self, text: str) -> Dict[str, Any]:
        lines = text.strip().splitlines()
        if len(lines) < 2:
            raise ValueError("CSV must have a header row and at least one data row")
        headers = [h.strip() for h in lines[0].split(",")]
        values  = [v.strip() for v in lines[1].split(",")]
        return dict(zip(headers, values))


# ── Cannot instantiate abstract class ────────────────────────────────────────
try:
    Parser()
except TypeError as exc:
    print("Cannot instantiate abstract class:", exc)

# ── JsonParser ────────────────────────────────────────────────────────────────
jp = JsonParser()
print(jp.safe_parse('{"key": "value", "n": 42}'))
print(jp.safe_parse('[1, 2, 3]'))           # wrong root type
print(jp.safe_parse('{bad json}'))          # malformed

# ── CsvParser ─────────────────────────────────────────────────────────────────
cp = CsvParser()
print(cp.safe_parse("name,score\nAlice,92"))
print(cp.safe_parse("only_header"))         # missing data row

Cannot instantiate abstract class: Can't instantiate abstract class Parser with abstract method parse
{'key': 'value', 'n': 42}
{'error': 'Expected dict, got list', 'raw': '[1, 2, 3]'}
{'error': 'Expecting property name enclosed in double quotes: line 1 column 2 (char 1)', 'raw': '{bad json}'}
{'name': 'Alice', 'score': '92'}
{'error': 'CSV must have a header row and at least one data row', 'raw': 'only_header'}


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">10) functools.lru_cache — Memoization</h2>
<p style="font-size: 15px;">
<code>@lru_cache</code> caches the return value of a pure function keyed on its arguments. Inspect with <code>.cache_info()</code> and reset with <code>.cache_clear()</code>. Only use on deterministic (side-effect-free) functions.
</p>
</div>

In [15]:
# ── functools.lru_cache: Memoization ─────────────────────────────────────────

@lru_cache(maxsize=128)
def fibonacci(n: int) -> int:
    """Recursive Fibonacci — without caching this is O(2^n)."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)

@lru_cache(maxsize=64)
def power(base: float, exp: int) -> float:
    return base ** exp

# ── Results ───────────────────────────────────────────────────────────────────
print("fib(0):", fibonacci(0))
print("fib(1):", fibonacci(1))
print("fib(10):", fibonacci(10))
print("fib(35):", fibonacci(35))
print("Cache info:", fibonacci.cache_info())

# Second call for same arg → cache hit (hits counter goes up)
_ = fibonacci(35)
print("After repeat call:", fibonacci.cache_info())

# Clearing the cache
fibonacci.cache_clear()
print("After cache_clear:", fibonacci.cache_info())
print("fib(10) recomputed:", fibonacci(10))
print("After recomputed:", fibonacci.cache_info())

# ── Edge cases ────────────────────────────────────────────────────────────────
try:
    fibonacci(-1)
except ValueError as exc:
    print("Edge case:", exc)

print("power(2, 10):", power(2, 10))
print("power cache:", power.cache_info())

fib(0): 0
fib(1): 1
fib(10): 55
fib(35): 9227465
Cache info: CacheInfo(hits=36, misses=36, maxsize=128, currsize=36)
After repeat call: CacheInfo(hits=37, misses=36, maxsize=128, currsize=36)
After cache_clear: CacheInfo(hits=0, misses=0, maxsize=128, currsize=0)
fib(10) recomputed: 55
After recomputed: CacheInfo(hits=8, misses=11, maxsize=128, currsize=11)
Edge case: n must be non-negative
power(2, 10): 1024
power cache: CacheInfo(hits=0, misses=1, maxsize=64, currsize=1)


<div style="font-family: 'Segoe UI', Calibri, sans-serif; font-size: 16px; line-height: 1.55;">
<h2 style="font-size: 22px;">11) Bonus — dataclass + Protocol (Modular Design)</h2>
<p style="font-size: 15px;">
<code>@dataclass</code> auto-generates <code>__init__</code>, <code>__repr__</code>, and <code>__eq__</code>. <code>Protocol</code> (structural subtyping) defines interfaces without requiring explicit inheritance — ideal for loose coupling.
</p>
</div>

In [13]:
@dataclass(frozen=True)
class Student:
    name: str
    age: int


class Formatter(Protocol):
    def format(self, student: Student) -> str:
        ...


class SimpleFormatter:
    def format(self, student: Student) -> str:
        return f"{student.name} ({student.age})"


def render_student(student: Student, formatter: Formatter) -> str:
    return formatter.format(student)


print(render_student(Student("Leena", 21), SimpleFormatter()))

Leena (21)
